In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .config("spark.driver.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.eventLog.gcMetrics.youngGenerationGarbageCollectors", "G1 Concurrent GC") \
    .config("spark.eventLog.gcMetrics.oldGenerationGarbageCollectors", "G1 Concurrent GC") \
    .config("spark.memory.offHeap.use", True)\
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
24/08/21 12:36:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
24/08/21 12:36:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
24/08/21 12:36:09 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
import pandas as pd
import numpy as np
import os
import pyotp
import pyspark 
from datetime import datetime, date, time, timedelta
import datetime as dt
from datetime import datetime
import json
import time
import requests
# import pyspark.pandas as ps
from IPython.display import clear_output

# importing module 

# importing sparksession from 
# pyspark.sql module 
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DateType, DoubleType, TimestampType, LongType, FloatType
from datetime import datetime
from pyspark.sql.functions import desc, asc, like, lit, abs, col, to_date, round
import pyspark.sql.functions as f
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
# import pyspark.sql.functions as sf
from pyspark.sql.functions import monotonically_increasing_id
# import matplotlib.pyplot as plt

from kiteconnect import KiteConnect
from kiteconnect import KiteTicker
from dotenv import load_dotenv
load_dotenv()  # Load credentials from .env file (not committed to version control)
creds = {
    'user_id': os.getenv('ZERODHA_USER_ID'),
    'password': os.getenv('ZERODHA_PASSWORD'),
    'totp_key': os.getenv('ZERODHA_TOTP_KEY'),
    'api_key': os.getenv('ZERODHA_API_KEY'),
    'api_secret': os.getenv('ZERODHA_API_SECRET')
}
base_url = "https://kite.zerodha.com"
login_url = "https://kite.zerodha.com/api/login"
twofa_url = "https://kite.zerodha.com/api/twofa"
instruments_url = "https://api.kite.trade/instruments"

session = requests.Session()
response = session.post(login_url,data={'user_id':creds['user_id'],'password':creds['password']})
request_id = json.loads(response.text)['data']['request_id']
twofa_pin = pyotp.TOTP(creds['totp_key']).now()
# twofa_pin =149166
response_1 = session.post(twofa_url,data={'user_id':creds['user_id'],'request_id':request_id,'twofa_value':twofa_pin,'twofa_type':'totp'})
kite = KiteConnect(api_key=creds['api_key'])
kite_url = kite.login_url()
try:
    session.get(kite_url)
except Exception as e:
    e_msg = str(e)
#print(e_msg)
request_token = e_msg.split('request_token=')[1].split(' ')[0].split('&action')[0]
print('Successful Login with Request Token:{}'.format(request_token))
data = kite.generate_session(request_token,creds['api_secret'])
access_token =  data['access_token']

kite.set_access_token(access_token)
#Get last traded prices for Nifty and BankNifty
# Initialize Kite Ticker
kws = KiteTicker(creds['api_key'], access_token)


Successful Login with Request Token:PQTCQpBB0e9W27lyAsafc2MYQHPTJFUN


In [3]:

# creating sparksession and giving 
# an app name 
spark = SparkSession.builder.appName('sparkdf').getOrCreate() 

# Define the schema for the DataFrame
schema = StructType([
    StructField("instrument_token", IntegerType(), nullable=False),
    StructField("exchange_token", StringType(), nullable=False),
    StructField("tradingsymbol", StringType(), nullable=False),
    StructField("name", StringType(), nullable=False),
    StructField("last_price", DoubleType(), nullable=False),
    StructField("expiry", DateType(), nullable=False),
    StructField("strike", DoubleType(), nullable=False),
    StructField("tick_size", DoubleType(), nullable=False),
    StructField("lot_size", IntegerType(), nullable=False),
    StructField("instrument_type", StringType(), nullable=False),
    StructField("segment", StringType(), nullable=False),
    StructField("exchange", StringType(), nullable=False)
])

# Read Parquet files into a DataFrame
Banknifty_Options_DF_From_File = spark.read.parquet("DataFiles/Instruments/Banknifty_Options", schema=schema)
Banknifty_Futures_DF_From_File = spark.read.parquet("DataFiles/Instruments/Banknifty_Futures", schema=schema)
Nifty_Options_DF_From_File = spark.read.parquet("DataFiles/Instruments/Nifty_Options", schema=schema)
Nifty_Futures_DF_From_File = spark.read.parquet("DataFiles/Instruments/Nifty_Futures", schema=schema)

# Define the schema for the expiry DataFrame
schema = StructType([
    StructField("name", StringType(), nullable=False),
    StructField("current_week_expiry", DateType(), nullable=False),
    StructField("current_month_expiry", DateType(), nullable=False),
    StructField("next_week_expiry", DateType(), nullable=False),
    StructField("next_month_expiry", DateType(), nullable=False)
])
Nifty_Banknifty_Expiries_DF = spark.read.parquet("DataFiles/Instruments/Nifty_Banknifty_Expiries", schema=schema)


Banknifty_current_week_expiry   = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("current_week_expiry").collect()[0][0]
Banknifty_current_month_expiry  = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("current_month_expiry").collect()[0][0]
Banknifty_next_week_expiry      = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("next_week_expiry").collect()[0][0]
Banknifty_next_month_expiry     = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "BANKNIFTY").select("next_month_expiry").collect()[0][0]


Nifty_current_week_expiry       = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("current_week_expiry").collect()[0][0]
Nifty_current_month_expiry      = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("current_month_expiry").collect()[0][0]
Nifty_next_week_expiry          = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("next_week_expiry").collect()[0][0]
Nifty_next_month_expiry         = Nifty_Banknifty_Expiries_DF.filter(Nifty_Banknifty_Expiries_DF['name'] == "NIFTY").select("next_month_expiry").collect()[0][0]


24/08/21 12:36:12 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
24/08/21 12:36:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(PS Scavenge), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
24/08/21 12:36:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(PS MarkSweep, PS Scavenge), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


In [4]:
Banknifty_Options_DF_From_File.select("expiry").distinct().orderBy("expiry").show()

+----------+
|    expiry|
+----------+
|2024-08-14|
|2024-08-21|
|2024-08-28|
|2024-09-04|
|2024-09-11|
|2024-09-18|
|2024-09-25|
|2024-10-30|
|2024-12-24|
|2025-03-26|
|2025-06-25|
+----------+



In [5]:
        
def get_Strike_OHLC_data(instrument_token,Name, strike, Expiry, CE_OR_PE, interval, No_Of_Days=1, No_Of_Days_Back=0):
        schema = StructType([
            StructField("date",     TimestampType(),    nullable=True),
            StructField("open",     FloatType(),        nullable=True),
            StructField("high",     FloatType(),        nullable=True),
            StructField("low",      FloatType(),        nullable=True),
            StructField("close",    FloatType(),        nullable=True),
            StructField("volume",   IntegerType(),      nullable=True),
            StructField("oi",       FloatType(),      nullable=True),
            StructField("day",      DateType(),         nullable=True)
        ])
        DF =  spark.read.parquet("DataFiles/HistoricalData/"+interval+"/"+str(Expiry)+"/"+str(instrument_token), schema=schema)\
            .filter(col("volume") > 0)\
            .withColumn("strike",lit(strike))\
            .distinct()
        DF.createOrReplaceTempView("DF")
        DF = spark.sql("""
            SELECT * FROM (
                SELECT  date,
                        open,
                        high,
                        low,
                        close,
                        oi,
                        day,
                        ((dense_rank() over (partition by strike order by day DESC)) - """ + str(No_Of_Days_Back) + """) as day_num,
                        strike
                FROM DF
                Order by date desc
            ) P WHERE close > 0 and day_num > 0 and day_num <= """ + str(No_Of_Days) + """
        """)
        return DF.orderBy(asc("date"))\
            .withColumn("instrument_token",lit(instrument_token))\
            .withColumn("name",lit(Name))\
            .withColumn("expiry",lit(Expiry))\
            .withColumn("instrument_type",lit(CE_OR_PE))\
            .withColumn("ohlc4",round((col('open')+col('high')+col('low')+col('close'))/4,2)).distinct()

In [6]:
def get_ButterFly_Data(Options_DF, Name, Cur_Expiry, Hedge_Expiry, CE_Strike, PE_Strike, Hedge_CE_Strike, Hedge_PE_Strike, inverted=False, interval="3minute", No_Of_Days=1, No_Of_Days_Back=0):
    if(inverted):
        ##Selling Instruments
        Hedge_CE_Strike_Token = Options_DF.filter(Options_DF['name'] == Name).filter(Options_DF['expiry'] == Cur_Expiry)\
        .filter(Options_DF['strike'] == CE_Strike).filter(Options_DF['instrument_type'] == 'CE').collect()[0]["instrument_token"]
        Hedge_PE_Strike_Token = Options_DF.filter(Options_DF['name'] == Name).filter(Options_DF['expiry'] == Cur_Expiry)\
        .filter(Options_DF['strike'] == PE_Strike).filter(Options_DF['instrument_type'] == 'PE').collect()[0]["instrument_token"]
        ##Byuing Instruments
        CE_Strike_Token = Options_DF.filter(Options_DF['name'] == Name).filter(Options_DF['expiry'] == Hedge_Expiry)\
        .filter(Options_DF['strike'] == Hedge_CE_Strike).filter(Options_DF['instrument_type'] == 'CE').collect()[0]["instrument_token"]
        PE_Strike_Token = Options_DF.filter(Options_DF['name'] == Name).filter(Options_DF['expiry'] == Hedge_Expiry)\
        .filter(Options_DF['strike'] == Hedge_PE_Strike).filter(Options_DF['instrument_type'] == 'PE').collect()[0]["instrument_token"]
    else:
        ##Selling Instruments
        CE_Strike_Token = Options_DF.filter(Options_DF['name'] == Name).filter(Options_DF['expiry'] == Cur_Expiry)\
        .filter(Options_DF['strike'] == CE_Strike).filter(Options_DF['instrument_type'] == 'CE').collect()[0]["instrument_token"]
        PE_Strike_Token = Options_DF.filter(Options_DF['name'] == Name).filter(Options_DF['expiry'] == Cur_Expiry)\
        .filter(Options_DF['strike'] == PE_Strike).filter(Options_DF['instrument_type'] == 'PE').collect()[0]["instrument_token"]
        ##Byuing Instruments
        Hedge_CE_Strike_Token = Options_DF.filter(Options_DF['name'] == Name).filter(Options_DF['expiry'] == Hedge_Expiry)\
        .filter(Options_DF['strike'] == Hedge_CE_Strike).filter(Options_DF['instrument_type'] == 'CE').collect()[0]["instrument_token"]
        Hedge_PE_Strike_Token = Options_DF.filter(Options_DF['name'] == Name).filter(Options_DF['expiry'] == Hedge_Expiry)\
        .filter(Options_DF['strike'] == Hedge_PE_Strike).filter(Options_DF['instrument_type'] == 'PE').collect()[0]["instrument_token"]
        
        CE_DF = get_Strike_OHLC_data(instrument_token = CE_Strike_Token,Name=Name, strike=CE_Strike, Expiry=Cur_Expiry, CE_OR_PE='CE',\
                             interval = interval, No_Of_Days=No_Of_Days, No_Of_Days_Back=No_Of_Days_Back)
        PE_DF = get_Strike_OHLC_data(instrument_token = PE_Strike_Token,Name=Name, strike=PE_Strike, Expiry=Cur_Expiry, CE_OR_PE='PE',\
                             interval = interval, No_Of_Days=No_Of_Days, No_Of_Days_Back=No_Of_Days_Back)
        
        Hedge_Expiry_CE_DF = get_Strike_OHLC_data(instrument_token = Hedge_CE_Strike_Token,Name=Name, strike=Hedge_CE_Strike, Expiry=Hedge_Expiry, CE_OR_PE='CE',\
                             interval = interval, No_Of_Days=No_Of_Days, No_Of_Days_Back=No_Of_Days_Back)
        Hedge_Expiry_PE_DF = get_Strike_OHLC_data(instrument_token = Hedge_PE_Strike_Token,Name=Name, strike=Hedge_PE_Strike, Expiry=Hedge_Expiry, CE_OR_PE='PE',\
                             interval = interval, No_Of_Days=No_Of_Days, No_Of_Days_Back=No_Of_Days_Back)
        
        Cur_Expiry_CE_PE_DF = CE_DF.union(PE_DF).withColumn("expiry",lit(Cur_Expiry))
        Cur_Expiry_CE_PE_DF.createOrReplaceTempView("CE_PE_DF")
        # Run SQL commands
        Cur_Expiry_CE_PE_DF = spark.sql("""
            SELECT * FROM (
                SELECT date,
                    round(sum(Open),2) As open,
                    round(sum(high),2) As high,
                    round(sum(low),2) As low,
                    round(sum(close),2) As close,
                    round(sum(case when instrument_type = 'CE' then close else null end),2)  As CE_close,
                    round(sum(case when instrument_type = 'PE' then close else null end),2)  As PE_close,
                    round(sum(oi),2) As oi,
                    round(sum(case when instrument_type = 'CE' then oi else null end),2)  As CE_oi,
                    round(sum(case when instrument_type = 'PE' then oi else null end),2)  As PE_oi,
                    day,
                    round(sum(case when instrument_type = 'CE' then ohlc4 else null end),2)  As CE_ohlc4,
                    round(sum(case when instrument_type = 'PE' then ohlc4 else null end),2)  As PE_ohlc4,
                    expiry,
                    ((dense_rank() over (partition by expiry order by day DESC)) - """ + str(No_Of_Days_Back) + """) as day_num
                FROM CE_PE_DF
                GROUP BY day, date, expiry
                Order by date desc
            ) P WHERE day_num > 0 and day_num <= """ + str(No_Of_Days) + """
                AND CE_close > 0 and PE_close > 0
        """)
        instrument_token = str(CE_Strike_Token) + ", " + str(PE_Strike_Token)
        Cur_Expiry_CE_PE_DF = Cur_Expiry_CE_PE_DF.orderBy(asc("date"))\
            .withColumn("instrument_token",lit(instrument_token))\
            .withColumn("CE_strike",lit(CE_Strike))\
            .withColumn("PE_strike",lit(PE_Strike))\
            .withColumn("ohlc4",round((col('open')+col('high')+col('low')+col('close'))/4,2))\
            .withColumn("Type",lit("Short")).distinct().orderBy("date")


        #hedge
        Hedge_Expiry_CE_PE_DF = Hedge_Expiry_CE_DF.union(Hedge_Expiry_PE_DF).withColumn("expiry",lit(Hedge_Expiry))
        Hedge_Expiry_CE_PE_DF.createOrReplaceTempView("Hedge_Expiry_CE_PE_DF")
        # Run SQL commands
        Hedge_Expiry_CE_PE_DF = spark.sql("""
            SELECT * FROM (
                SELECT date,
                    round(sum(Open),2) As open,
                    round(sum(high),2) As high,
                    round(sum(low),2) As low,
                    round(sum(close),2) As close,
                    round(sum(case when instrument_type = 'CE' then close else null end),2)  As CE_close,
                    round(sum(case when instrument_type = 'PE' then close else null end),2)  As PE_close,
                    round(sum(oi),2) As oi,
                    round(sum(case when instrument_type = 'CE' then oi else null end),2)  As CE_oi,
                    round(sum(case when instrument_type = 'PE' then oi else null end),2)  As PE_oi,
                    day,
                    round(sum(case when instrument_type = 'CE' then ohlc4 else null end),2)  As CE_ohlc4,
                    round(sum(case when instrument_type = 'PE' then ohlc4 else null end),2)  As PE_ohlc4,
                    expiry,
                    ((dense_rank() over (partition by expiry order by day DESC)) - """ + str(No_Of_Days_Back) + """) as day_num
                FROM Hedge_Expiry_CE_PE_DF
                GROUP BY day, date, expiry
                Order by date desc
            ) P WHERE day_num > 0 and day_num <= """ + str(No_Of_Days) + """
                AND CE_close > 0 and PE_close > 0
        """)
        instrument_token = str(Hedge_CE_Strike_Token) + ", " + str(Hedge_PE_Strike_Token)
        Hedge_Expiry_CE_PE_DF = Hedge_Expiry_CE_PE_DF\
            .withColumn("instrument_token",lit(instrument_token))\
            .withColumn("CE_strike",lit(Hedge_CE_Strike))\
            .withColumn("PE_strike",lit(Hedge_PE_Strike))\
            .withColumn("ohlc4",round((col('open')+col('high')+col('low')+col('close'))/4,2))\
            .withColumn("Type",lit("Long")).distinct().orderBy("date")
        clear_output()
        
        DF = Cur_Expiry_CE_PE_DF.union(Hedge_Expiry_CE_PE_DF).withColumn("expiry",lit(Cur_Expiry)).orderBy("date")
    #return [CE_Strike_Token,PE_Strike_Token, Hedge_CE_Strike_Token, Hedge_PE_Strike_Token]
    return DF

In [7]:

def get_ROC_added(df, col_name = "close",length = 20):
    df = df.withColumn("ROC_AVG_"+col_name,lit(0))
    columns_to_drop = []
    for i in range(1,length+1):
        df = df.withColumn("ROC_"+str(i), (col(col_name) - lag(col(col_name), i).over(Window.partitionBy("instrument_token").orderBy("rownum"))) / lag(col(col_name), i).over(Window.partitionBy("instrument_token").orderBy("rownum")))
        df = df.withColumn("ROC_AVG_"+col_name, (col("ROC_AVG_"+col_name) + col("ROC_"+str(i))))
        columns_to_drop.append("ROC_"+str(i))
    df = df.withColumn("ROC_AVG_"+col_name, col("ROC_AVG_"+col_name)/2)
    # df = df.drop("column1", "column2", "column3")
    df = df.drop(*columns_to_drop)
    return df
    

In [8]:
def round_to_hundred(num):
    if num % 100 >= 50:
        return num + (100 - num % 100)
    else:
        return num - (num % 100)
    
def round_to_fifty(num):
    if num % 50 >= 25:
        return num + (50 - num % 50)
    else:
        return num - (num % 50)
    

# def get_ATM_Strike(instrument_token, ronding_number):
#     _ltp = (kite.ltp(instrument_token)[str(instrument_token)]["last_price"])
#     #get ATM Strikes
#     if ronding_number == 100:
#         return round_to_hundred(_ltp)
#     if ronding_number == 50:
#         return round_to_fifty(_ltp)
    


In [9]:
interval = "3minute"
from pyspark.sql.functions import col, lag,when
from pyspark.sql.window import Window
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [10]:
spark.sql("""
        SELECT CAST(NOW() AS DATE) AS DATE """)

DataFrame[DATE: date]

In [11]:
def getchart(Calendar_DF, IS_latest_Day=False):
    Calendar_DF.createOrReplaceTempView("Calendar_DF")
    # Run SQL commands
    # spark.sql("""(SELECT distinct day, day_num FROM Calendar_DF)""").show()
    Calendar_DF1 = spark.sql("""
        SELECT *, (CASE WHEN day_num = (SELECT MIN(day_num) FROM Calendar_DF) THEN 1 ELSE 0 END) AS IS_LATEST_DAY FROM (
            SELECT day,
                date,
                max((case when Type = 'Short'  then instrument_token else null end) + ', ' + 
                            (case when Type = 'Long'  then instrument_token else null end)) AS instrument_token,
                round(sum(case when Type = 'Short'  then close else null end),2)  As Short_close,
                round(sum(case when Type = 'Long'   then close else null end),2)  As Long_close,     
                round((sum(case when Type = 'Short'  then close else null end) - sum(case when Type = 'Long'   then close else null end)),2) As Close_Diff1,
                round((sum(case when Type = 'Short'  then close else null end) + sum(case when Type = 'Long'   then close else null end)),2) As Close_Diff2,
                round(sum(case when Type = 'Short'  then round((oi),2) else null end),2)  As Short_oi,
                round(sum(case when Type = 'Long'   then round((oi),2) else null end),2)  As Long_oi,
                round(sum(case when Type = 'Short'  then round((ohlc4),2) else null end),2)  As Short_ohlc4,
                round(sum(case when Type = 'Long'   then round((ohlc4),2) else null end),2)  As Long_ohlc4,
                expiry,
                day_num
            FROM Calendar_DF
            GROUP BY day, date, expiry, day_num
            Order by date asc
        ) P WHERE day_num > 0
    """)

    Calendar_DF1 = Calendar_DF1.withColumn("MidPoint", lit(0)) 
    # Calendar_DF1 = Calendar_DF1.filter(Calendar_DF["date"].isNotNull()).isNotNull().orderBy("date")
    Calendar_DF1 = Calendar_DF1.withColumn("rownum", monotonically_increasing_id())
    Calendar_DF1 = get_ROC_added(Calendar_DF1, col_name = "Short_close",length = 20)
    Calendar_DF1 = get_ROC_added(Calendar_DF1, col_name = "Long_close",length = 20)
    Calendar_DF1 = get_ROC_added(Calendar_DF1, col_name = "Close_Diff1",length = 20)
    Calendar_DF1 = get_ROC_added(Calendar_DF1, col_name = "Close_Diff2",length = 20)

    if IS_latest_Day :
        Calendar_DF1 = Calendar_DF1.filter(Calendar_DF1['IS_LATEST_DAY'] == True)
        Calendar_DF1 = Calendar_DF1.withColumn("rownum", monotonically_increasing_id())
    Calendar_DF1 = Calendar_DF1.toPandas()
    Calendar_DF1['date']    = pd.to_datetime(Calendar_DF1['date'])
    Calendar_DF1['rownum']  = pd.to_numeric(Calendar_DF1['rownum'])
    Calendar_DF1['Short_close']    = pd.to_numeric(Calendar_DF1['Short_close'])
    Calendar_DF1['Long_close']    = pd.to_numeric(Calendar_DF1['Long_close'])
    Calendar_DF1['Close_Diff1']     = pd.to_numeric(Calendar_DF1['Close_Diff1'])
    Calendar_DF1['Close_Diff2']     = pd.to_numeric(Calendar_DF1['Close_Diff2'])

    # Calendar_DF1.show()

    fig = make_subplots(rows=2, cols=2)
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['Short_close'], mode='lines', name='Short close', line=dict(color='green', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['Long_close'], mode='lines', name='Long close', line=dict(color='red', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['Close_Diff1'], mode='lines', name='Close Diff 1 (L+S)', line=dict(color='black', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['Close_Diff2'], mode='lines', name='Close Diff 2 (L+S)', line=dict(color='black', width=2)),
        row=1, col=2
    )


    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['MidPoint'], mode='lines', name='MidLine'),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['ROC_AVG_Short_close'], mode='lines', name='Short Close MMI', line=dict(color='green', width=2)),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['ROC_AVG_Long_close'], mode='lines', name='Long Close MMI', line=dict(color='red', width=2)),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['ROC_AVG_Close_Diff1'], mode='lines', name='Close Diff 1 MMI', line=dict(color='black', width=2)),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['MidPoint'], mode='lines', name='MidLine'),
        row=2, col=2
    )
    fig.add_trace(
        go.Scatter(x=Calendar_DF1['rownum'], y=Calendar_DF1['ROC_AVG_Close_Diff2'], mode='lines', name='Close Diff 2 MMI', line=dict(color='black', width=2)),
        row=2, col=2
    )

    # Update the layout and show the figure
    fig.update_layout(height=600, width=1200, title_text=Short_Chart_Title + "      Data Pulled At : " + str(datetime.now().strftime("%Y-%m-%d %I:%M %p")))
    fig.update_layout(paper_bgcolor='gray')
    fig.update_layout(margin=dict(l=50, r=50, b=50, t=50))
    fig.show()



In [12]:

Banknifty_current_week_expiry   = "2024-08-21"
Banknifty_next_week_expiry      = "2024-08-21"
CE_Strike                       = 50900
PE_Strike                       = 50800
Hedge_CE_Strike                 = 51200
Hedge_PE_Strike                 = 50400
inverted                        = False
interval                        = "3minute"
No_Of_Days                      = 2 #if you want latest data keep this 2 and set IS_latest_Day = True
No_Of_Days_Back                 = 0
IS_latest_Day                   = True

Short_Chart_Title= ("Short CE_Strike : "  + str(CE_Strike)       + " | Short PE_Strike : "   + str(PE_Strike))
Long_Chart_Title = ("Long CE_Strike : "   + str(Hedge_CE_Strike) + " | Long PE_Strike : "    + str(Hedge_CE_Strike))
# print(Short_Chart_Title)
# Calendar_DF = get_ButterFly_Data(Options_DF = Banknifty_Options_DF_From_File, Name="BANKNIFTY", \
#             Cur_Expiry=Banknifty_current_week_expiry, Hedge_Expiry=Banknifty_next_week_expiry,\
#             CE_Strike=CE_Strike, PE_Strike=PE_Strike, Hedge_CE_Strike=Hedge_CE_Strike, Hedge_PE_Strike=Hedge_PE_Strike,\
#             inverted=inverted, interval=interval, No_Of_Days=No_Of_Days, No_Of_Days_Back=No_Of_Days_Back)

clear_output()
Calendar_DF = get_ButterFly_Data(Options_DF = Banknifty_Options_DF_From_File, Name="BANKNIFTY", \
    Cur_Expiry=Banknifty_current_week_expiry, Hedge_Expiry=Banknifty_next_week_expiry,\
    CE_Strike=CE_Strike, PE_Strike=PE_Strike, Hedge_CE_Strike=Hedge_CE_Strike, Hedge_PE_Strike=Hedge_PE_Strike,\
    inverted=inverted, interval=interval, No_Of_Days=No_Of_Days, No_Of_Days_Back=No_Of_Days_Back)
getchart(Calendar_DF, IS_latest_Day)

while True:
    try:
        clear_output()
        Calendar_DF = get_ButterFly_Data(Options_DF = Banknifty_Options_DF_From_File, Name="BANKNIFTY", \
            Cur_Expiry=Banknifty_current_week_expiry, Hedge_Expiry=Banknifty_next_week_expiry,\
            CE_Strike=CE_Strike, PE_Strike=PE_Strike, Hedge_CE_Strike=Hedge_CE_Strike, Hedge_PE_Strike=Hedge_PE_Strike,\
            inverted=inverted, interval=interval, No_Of_Days=No_Of_Days, No_Of_Days_Back=No_Of_Days_Back)
        getchart(Calendar_DF, IS_latest_Day)
        time.sleep(60)
    except Exception as e:
        print(f"An error occurred: {e}")
        continue
    else:
        continue


KeyboardInterrupt: 

In [ ]:
getchart(Calendar_DF, IS_latest_Day)

In [ ]:
getchart(Calendar_DF)

In [ ]:
# #Long
# # Calendar_DF.show()
# Long_DF = Calendar_DF.filter(col("Type") == "Long").filter(Calendar_DF["date"].isNotNull()).filter(Calendar_DF["close"].isNotNull()).orderBy("date")
# Long_DF = Long_DF.withColumn("rownum", monotonically_increasing_id())
# Long_DF = get_ROC_added(Long_DF, col_name = "close",length = 20)

# # Define a window for calculating the cumulative sum and count
# w = Window.partitionBy("instrument_token").orderBy('date').rowsBetween(Window.unboundedPreceding, 0)
# # Create a new DataFrame with columns for cumulative sum and count
# Long_DF = Long_DF.withColumn('cumulative_sum_close', F.sum('close').over(w))
# Long_DF = Long_DF.withColumn('cumulative_min_close', F.min('close').over(w))
# Long_DF = Long_DF.withColumn('cumulative_count_close', F.count('close').over(w))
# Long_DF = Long_DF.withColumn('cumulative_sum_oi', F.sum('oi').over(w))
# Long_DF = Long_DF.withColumn('cumulative_count_oi', F.count('oi').over(w))
# # Calculate the incremental average
# Long_DF = Long_DF.withColumn('incremental_close_avg', Long_DF['cumulative_sum_close'] / Long_DF['cumulative_count_close'])
# Long_DF = Long_DF.withColumn('incremental_oi_avg', Long_DF['cumulative_sum_oi'] / Long_DF['cumulative_count_oi'])
# Long_DF = Long_DF.withColumn('incremental_min_close', Long_DF['cumulative_min_close'])
# Long_DF = Long_DF.drop("cumulative_sum_close", "colucumulative_min_closemn2", "cumulative_count_close", "cumulative_sum_oi", "cumulative_count_oi")
# Long_DF = Long_DF.withColumn("MidPoint", lit(0)) 

# Long_PD_DF = Long_DF.toPandas()
# Long_PD_DF['date'] = pd.to_datetime(Long_PD_DF['date'])
# Long_PD_DF['rownum'] = pd.to_numeric(Long_PD_DF['rownum'])
# Long_PD_DF['open'] = pd.to_numeric(Long_PD_DF['open'])
# Long_PD_DF['high'] = pd.to_numeric(Long_PD_DF['high'])
# Long_PD_DF['low'] = pd.to_numeric(Long_PD_DF['low'])
# Long_PD_DF['close'] = pd.to_numeric(Long_PD_DF['close'])

# #Short
# Short_DF = Calendar_DF.filter(col("Type") == "Short").filter(Calendar_DF["date"].isNotNull()).filter(Calendar_DF["close"].isNotNull()).orderBy("date")
# Short_DF = Short_DF.withColumn("rownum", monotonically_increasing_id())
# Short_DF = get_ROC_added(Short_DF, col_name = "close",length = 20)

# # Define a window for calculating the cumulative sum and count
# w = Window.partitionBy("instrument_token").orderBy('date').rowsBetween(Window.unboundedPreceding, 0)
# # Create a new DataFrame with columns for cumulative sum and count
# Short_DF = Short_DF.withColumn('cumulative_sum_close', F.sum('close').over(w))
# Short_DF = Short_DF.withColumn('cumulative_min_close', F.min('close').over(w))
# Short_DF = Short_DF.withColumn('cumulative_count_close', F.count('close').over(w))
# Short_DF = Short_DF.withColumn('cumulative_sum_oi', F.sum('oi').over(w))
# Short_DF = Short_DF.withColumn('cumulative_count_oi', F.count('oi').over(w))
# # Calculate the incremental average
# Short_DF = Short_DF.withColumn('incremental_close_avg', Short_DF['cumulative_sum_close'] / Short_DF['cumulative_count_close'])
# Short_DF = Short_DF.withColumn('incremental_oi_avg', Short_DF['cumulative_sum_oi'] / Short_DF['cumulative_count_oi'])
# Short_DF = Short_DF.withColumn('incremental_min_close', Short_DF['cumulative_min_close'])
# Short_DF = Short_DF.drop("cumulative_sum_close", "colucumulative_min_closemn2", "cumulative_count_close", "cumulative_sum_oi", "cumulative_count_oi")
# Short_DF = Short_DF.withColumn("MidPoint", lit(0))

# Short_PD_DF = Short_DF.toPandas()
# Short_PD_DF['date'] = pd.to_datetime(Short_PD_DF['date'])
# Short_PD_DF['rownum'] = pd.to_numeric(Short_PD_DF['rownum'])
# Short_PD_DF['open'] = pd.to_numeric(Short_PD_DF['open'])
# Short_PD_DF['high'] = pd.to_numeric(Short_PD_DF['high'])
# Short_PD_DF['low'] = pd.to_numeric(Short_PD_DF['low'])
# Short_PD_DF['close'] = pd.to_numeric(Short_PD_DF['close'])



In [ ]:
# #Long
# # Calendar_DF.show()
# Long_DF = Calendar_DF.filter(col("Type") == "Long").filter(Calendar_DF["date"].isNotNull()).filter(Calendar_DF["close"].isNotNull()).orderBy("date")
# Long_DF = Long_DF.withColumn("rownum", monotonically_increasing_id())
# Long_DF = get_ROC_added(Long_DF, col_name = "close",length = 20)

# # Define a window for calculating the cumulative sum and count
# w = Window.partitionBy("instrument_token").orderBy('date').rowsBetween(Window.unboundedPreceding, 0)
# # Create a new DataFrame with columns for cumulative sum and count
# Long_DF = Long_DF.withColumn('cumulative_sum_close', F.sum('close').over(w))
# Long_DF = Long_DF.withColumn('cumulative_min_close', F.min('close').over(w))
# Long_DF = Long_DF.withColumn('cumulative_count_close', F.count('close').over(w))
# Long_DF = Long_DF.withColumn('cumulative_sum_oi', F.sum('oi').over(w))
# Long_DF = Long_DF.withColumn('cumulative_count_oi', F.count('oi').over(w))
# # Calculate the incremental average
# Long_DF = Long_DF.withColumn('incremental_close_avg', Long_DF['cumulative_sum_close'] / Long_DF['cumulative_count_close'])
# Long_DF = Long_DF.withColumn('incremental_oi_avg', Long_DF['cumulative_sum_oi'] / Long_DF['cumulative_count_oi'])
# Long_DF = Long_DF.withColumn('incremental_min_close', Long_DF['cumulative_min_close'])
# Long_DF = Long_DF.drop("cumulative_sum_close", "colucumulative_min_closemn2", "cumulative_count_close", "cumulative_sum_oi", "cumulative_count_oi")
# Long_DF = Long_DF.withColumn("MidPoint", lit(0)) 

# Long_PD_DF = Long_DF.toPandas()
# Long_PD_DF['date'] = pd.to_datetime(Long_PD_DF['date'])
# Long_PD_DF['rownum'] = pd.to_numeric(Long_PD_DF['rownum'])
# Long_PD_DF['open'] = pd.to_numeric(Long_PD_DF['open'])
# Long_PD_DF['high'] = pd.to_numeric(Long_PD_DF['high'])
# Long_PD_DF['low'] = pd.to_numeric(Long_PD_DF['low'])
# Long_PD_DF['close'] = pd.to_numeric(Long_PD_DF['close'])

# #Short
# Short_DF = Calendar_DF.filter(col("Type") == "Short").filter(Calendar_DF["date"].isNotNull()).filter(Calendar_DF["close"].isNotNull()).orderBy("date")
# Short_DF = Short_DF.withColumn("rownum", monotonically_increasing_id())
# Short_DF = get_ROC_added(Short_DF, col_name = "close",length = 20)

# # Define a window for calculating the cumulative sum and count
# w = Window.partitionBy("instrument_token").orderBy('date').rowsBetween(Window.unboundedPreceding, 0)
# # Create a new DataFrame with columns for cumulative sum and count
# Short_DF = Short_DF.withColumn('cumulative_sum_close', F.sum('close').over(w))
# Short_DF = Short_DF.withColumn('cumulative_min_close', F.min('close').over(w))
# Short_DF = Short_DF.withColumn('cumulative_count_close', F.count('close').over(w))
# Short_DF = Short_DF.withColumn('cumulative_sum_oi', F.sum('oi').over(w))
# Short_DF = Short_DF.withColumn('cumulative_count_oi', F.count('oi').over(w))
# # Calculate the incremental average
# Short_DF = Short_DF.withColumn('incremental_close_avg', Short_DF['cumulative_sum_close'] / Short_DF['cumulative_count_close'])
# Short_DF = Short_DF.withColumn('incremental_oi_avg', Short_DF['cumulative_sum_oi'] / Short_DF['cumulative_count_oi'])
# Short_DF = Short_DF.withColumn('incremental_min_close', Short_DF['cumulative_min_close'])
# Short_DF = Short_DF.drop("cumulative_sum_close", "colucumulative_min_closemn2", "cumulative_count_close", "cumulative_sum_oi", "cumulative_count_oi")
# Short_DF = Short_DF.withColumn("MidPoint", lit(0))

# Short_PD_DF = Short_DF.toPandas()
# Short_PD_DF['date'] = pd.to_datetime(Short_PD_DF['date'])
# Short_PD_DF['rownum'] = pd.to_numeric(Short_PD_DF['rownum'])
# Short_PD_DF['open'] = pd.to_numeric(Short_PD_DF['open'])
# Short_PD_DF['high'] = pd.to_numeric(Short_PD_DF['high'])
# Short_PD_DF['low'] = pd.to_numeric(Short_PD_DF['low'])
# Short_PD_DF['close'] = pd.to_numeric(Short_PD_DF['close'])



In [ ]:
def plot_Chart(Short_PD_DF, Long_PD_DF, Short_Chart_Title, Long_Chart_Title):
    fig = make_subplots(rows=2, cols=2)
    fig.add_trace(
        go.Scatter(x=Short_PD_DF['rownum'], y=Short_PD_DF['close'], mode='lines', name='Staddle Close', line=dict(color='black', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=Short_PD_DF['rownum'], y=Short_PD_DF['incremental_close_avg'], mode='lines', name='Staddle AVG', line=dict(color='gray', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=Short_PD_DF['rownum'], y=(Short_PD_DF['incremental_min_close']+Short_PD_DF['incremental_close_avg'])/2, mode='lines', name='Trailing SL', line=dict(color='red', width=2)),
        row=1, col=1
    )
    # Update the layout and show the figure
    fig.update_layout(height=600, width=1200, title_text=Short_Chart_Title + "      Data Pulled At : " + str(datetime.now().strftime("%Y-%m-%d %I:%M %p")))
    fig.update_layout(paper_bgcolor='gray')
    fig.update_layout(margin=dict(l=50, r=50, b=50, t=50))
    fig.show()


plot_Chart(Short_PD_DF, Long_PD_DF, Short_Chart_Title, Long_Chart_Title)

In [ ]:


def plot_Chart(df_pandas,strike):
    # Create a figure with two subplots

    
    if not(CE_OR_PE == 'CE' or CE_OR_PE == 'PE'):
        fig = make_subplots(rows=2, cols=2)
        fig2 = make_subplots(rows=1, cols=2)
    else:
        fig = make_subplots(rows=2, cols=1)
        fig2 = make_subplots(rows=1, cols=1)
        
    # Add the 'close' line chart to the first subplot
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['close'], mode='lines', name='Staddle Close', line=dict(color='black', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['incremental_close_avg'], mode='lines', name='Staddle AVG', line=dict(color='gray', width=2)),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=(df_pandas['incremental_min_close']+df_pandas['incremental_close_avg'])/2, mode='lines', name='Trailing SL', line=dict(color='red', width=2)),
        row=1, col=1
    )


    if not(CE_OR_PE == 'CE' or CE_OR_PE == 'PE'):
        # Add the 'CE close' line chart to the first subplot
        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['CE_close'], mode='lines', name='CALL', line=dict(color='red', width=2)),
            row=1, col=2
        )
        # Add the 'PE close' line chart to the first subplot
        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['PE_close'], mode='lines', name='PUT', line=dict(color='green', width=2)),
            row=1, col=2
        )

        
        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['MidPoint'], mode='lines', name='MidLine'),
            row=2, col=2
        )
        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_CE_close'], mode='lines', name='CE_ROC_AVG', line=dict(color='red', width=2)),
            row=2, col=2
        )

        fig.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_PE_close'], mode='lines', name='PE_ROC_AVG', line=dict(color='green', width=2)),
            row=2, col=2
        )
        # fig.add_trace(
        #     go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_CEPE_AVG'], mode='lines', name='CE_PE_ROC_AVG', line=dict(color='black', width=2)),
        #     row=2, col=2
        # )
    
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['MidPoint'], mode='lines', name='MidLine'),
        row=2, col=1
    )
    fig.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_close'], mode='lines', name='ROC_AVG', line=dict(color='black', width=2)),
        # go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_close'], mode="lines+text", name="ROC AVG", text=['ROC_AVG_close'],textposition="top center", line=dict(color='black', width=2)),
        row=2, col=1
    )
    # fig.add_trace(
    #     go.Scatter(x=df_pandas['rownum'], y=df_pandas['ROC_AVG_CEPE_AVG'], mode='lines', name='CE_PE_ROC_AVG', line=dict(color='gray', width=2)),
    #     row=2, col=1
    # )
    
    # vertical_line_x = [125]
    # fig.add_trace(go.Scatter(x=[vertical_line_x], y=[None, None], line_width=2, line_color='black', line_dash='dash'))

    # Add the 'open interest' line chart to the second subplot
    fig2.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['oi'], mode='lines', name='Combined OI'),
        row=1, col=1
    )

    fig2.add_trace(
        go.Scatter(x=df_pandas['rownum'], y=df_pandas['incremental_oi_avg'], mode='lines', name='Staddle OI AVG', line=dict(color='gray', width=2)),
        row=1, col=1
    )
    # trace3 = go.Scatter(
    #     x=x,
    #     y=y3,
    #     mode='lines',
    #     name='Tapjoy',
    #     line=dict(color='purple', width=2)
    # )
    if not(CE_OR_PE == 'CE' or CE_OR_PE == 'PE'):
        # Add the 'PE_OI' line chart to the second subplot
        fig2.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['CE_oi'], mode='lines', name='CALL OI'),
            row=1, col=2
        )
        # Add the 'CE_OI' line chart to the second subplot
        fig2.add_trace(
            go.Scatter(x=df_pandas['rownum'], y=df_pandas['PE_oi'], mode='lines', name='PUT OI'),
            row=1, col=2
        )
    # Update the layout and show the figure
    fig.update_layout(height=600, width=1200, title_text="For Stike Level: (" +Strike_level_Name + ") / " + str(strike) + "          Data Pulled At : " + str(datetime.now().strftime("%Y-%m-%d %I:%M %p")))
    fig.update_layout(paper_bgcolor='gray')
    fig.update_layout(margin=dict(l=50, r=50, b=50, t=50))
    fig.show()
    fig2.update_layout(height=400, width=1200, title_text="For Stike Level: (" +Strike_level_Name + ") / " + str(strike) + "          Data Pulled At : " + str(datetime.now().strftime("%Y-%m-%d %I:%M %p")))
    fig2.update_layout(paper_bgcolor='gray')
    fig2.show()

    fig3 = go.Figure(data=[go.Candlestick(x=df_pandas['date'],
                    open=df_pandas['open'],
                    high=df_pandas['high'],
                    low=df_pandas['low'],
                    close=df_pandas['close'])])

    fig3.update_layout(height=600, width=1200, title_text="For Stike Level: (" +Strike_level_Name + ") / " + str(strike) + "          Data Pulled At : " + str(datetime.now().strftime("%Y-%m-%d %H:%M")))
    fig3.update_layout(xaxis_rangeslider_visible=False)
    fig3.show()



In [ ]:
#Banknifty ATM Strike data

#For Banknifty Current Week Expiry
# Banknifty_Options_DF = get_Options_DF(Banknifty_Options_DF_From_File, get_ATM_Strike(Banknifty_instrument_token,100) ,Banknifty_current_week_expiry,100,9)

# #For Nifty Current Week Expiry
# Nifty_Options_DF = get_Options_DF(Nifty_Options_DF_From_File, get_ATM_Strike(Nifty_instrument_token,50), Nifty_current_week_expiry, 50,9)


BankNifty_Use_Custom_Strike=True
BankNifty_Custom_Strike=49300
Nifty_Use_Custom_Strike=True
Nifty_Custom_Strike=22950
How_Many_Strikes = 20

#For Banknifty Current Week Expiry
if BankNifty_Use_Custom_Strike:
    Banknifty_Options_DF = get_Options_DF(Banknifty_Options_DF_From_File, BankNifty_Custom_Strike ,Banknifty_current_week_expiry,100,How_Many_Strikes)
else:
    Banknifty_Options_DF = get_Options_DF(Banknifty_Options_DF_From_File, get_ATM_Strike(Banknifty_instrument_token,100) ,Banknifty_current_week_expiry,100,How_Many_Strikes)
#For Nifty Current Week Expiry
if Nifty_Use_Custom_Strike:
    Nifty_Options_DF = get_Options_DF(Nifty_Options_DF_From_File, Nifty_Custom_Strike, Nifty_current_week_expiry, 50,How_Many_Strikes)
else:
    Nifty_Options_DF = get_Options_DF(Nifty_Options_DF_From_File, get_ATM_Strike(Nifty_instrument_token,50), Nifty_current_week_expiry, 50,How_Many_Strikes)


In [ ]:
import schedule
import time
from IPython.display import clear_output

In [ ]:


#now get data for the token
Strike_level_Name   = 'ATM'
No_Of_Days          = 1
No_Of_Days_Back     = 0
CE_OR_PE            = "E"
def job():
    # Clear the output of the current cell
    clear_output()
    # print(" Data Pulled at : " + str(datetime.now()))
    get_chart()
job()
schedule.clear()
# Schedule the job to run every 3 minutes
schedule.every(1).minutes.do(job)


# # Start the job at 9:15 am each day
# schedule.every().day.at("09:15").do(job)

# Run the scheduled jobs
while True:
    try:
        schedule.run_pending()
        time.sleep(5)
    except Exception as e:
        print(f"An error occurred: {e}")
        continue
    else:
        continue

In [ ]:
#now get data for the token
Strike_level_Name   = 'ATM'
No_Of_Days          = 1
No_Of_Days_Back     = 0
CE_OR_PE            = "E"
Custom_expiry       = Banknifty_current_week_expiry#'2024-05-29'
# Custom_expiry       = datetime.strptime('2024-05-29', '%Y-%m-%d').date()

get_chart(Custom_expiry)